# Configurations

In [28]:
BOOTSTRAP_SERVER = "10.67.22.90:19092" # Kafka port 19092 is dedicated to VMs in CloudVeneto

TOPIC_STREAM = "topic_stream"
TOPIC_RESULTS = "topic_results"

SCHEDULER_IP = "10.67.22.90"
WORKER_IPS = ["10.67.22.58", "10.67.22.244", "10.67.22.210", "10.67.22.57"]
N_WORKERS = len(WORKER_IPS)
N_PARTITIONS = N_WORKERS * 2

WORKER_PUBLISH_INTERVAL_SEC = 5.0

# Topics setup

In [29]:
from kafka.admin import KafkaAdminClient
from kafka.errors import UnknownTopicOrPartitionError

print("Creating topics...")
# All settings documented in https://kafka-python.readthedocs.io/en/master/apidoc/KafkaAdminClient.html
kafka_admin = KafkaAdminClient(
    bootstrap_servers=BOOTSTRAP_SERVER,
    client_id = "kafka_admin_client",
    security_protocol = "PLAINTEXT"
)

# Delete old topics if they exist
try:
    kafka_admin.delete_topics([TOPIC_STREAM, TOPIC_RESULTS])
except UnknownTopicOrPartitionError:
    pass

# All settings documented in https://kafka.apache.org/42/configuration/topic-configs/
kafka_admin.create_topics({
    TOPIC_STREAM: {
        "num_partitions": N_PARTITIONS,
        "replication_factor": 1,
        "topic_configs": {
            "message.timestamp.type": "CreateTime", # use producer timestamp instead of log append timestamp
        },
    },
    TOPIC_RESULTS: {
        "num_partitions": 1,
        "replication_factor": 1,
        "topic_configs": {
            "message.timestamp.type": "CreateTime",
        },
    }
})

print("Waiting for topics...")
# Wait until all topics are ready to use
kafka_admin.wait_for_topics([TOPIC_STREAM, TOPIC_RESULTS])
print("Topics ready: ", kafka_admin.list_topics())

 # Close the KafkaAdminClient connection to the Kafka broker
kafka_admin.close()

Creating topics...
Waiting for topics...
Topics ready:  ['topic_results', 'topic_stream', '__consumer_offsets']


# Dask cluster setup

Dask needs to be installed in all nodes

Documentation: https://docs.dask.org/en/stable/deploying-ssh.html and https://docs.dask.org/en/stable/_modules/distributed/client.html

In [46]:
from dask.distributed import Client, SSHCluster

# Disable Dask info logs, keep only warnings and errors
import logging
logging.getLogger("distributed.deploy.ssh").setLevel(logging.WARN)

print("Creating Dask SSH cluster...")
dask_cluster = SSHCluster(
    [SCHEDULER_IP] + WORKER_IPS,
    connect_options = {"known_hosts": None},                              # Disable server host key validation
    worker_options = {"n_workers": 1, "nthreads": 1},                     # Options to pass on to workers
    scheduler_options={"port": 8786, "dashboard_address": "0.0.0.0:8797"} # Options to pass on to scheduler
)

dask_client = Client(dask_cluster, name="dask_client")
dask_client

Creating Dask SSH cluster...


<Client: 'tcp://10.67.22.90:8786' processes=2 threads=2, memory=3.83 GiB>

# Workers code

In [45]:
def worker_function(worker_id, bootstrap_servers, input_topic, output_topic):
    import json
    import time
    import numpy as np
    from kafka import KafkaConsumer, KafkaProducer

    consumer = KafkaConsumer(
        input_topic,
        bootstrap_servers=bootstrap_servers,
        client_id = f"worker{worker_id}_consumer", # Appears in server-side logs
        group_id = 'dask_processing_group',        # consumer group to join for dynamic partition assignment
        group_instance_id = f"worker{worker_id}_consumer_instance", # Make this consumer a static member of the group (not really necessary)
        value_deserializer = None,                 # Deserialization will be performed manually
        max_partition_fetch_bytes = 65*1024*1024,  # This size must be at least as large as the maximum message size the server allows
        auto_offset_reset='earliest',              # Bring the reading offset to the earliest message
        security_protocol = "PLAINTEXT", 
    )
    
    # Setup producer. This will publish data for the dashboard
    producer = KafkaProducer(
        bootstrap_servers=bootstrap_servers,
        client_id = f"worker{worker_id}_producer", # Appears in server-side logs
        value_serializer=lambda v: json.dumps(v).encode('utf-8'),
        compression_type=None,                     # No compression
        enable_idempotence = True,                 # Ensure that exactly one copy of each message is written in the stream
        batch_size = 16384,                         # Enable Kafka batching. Batch size is large so kafka will flush every linges_ms ms
        linger_ms = 5000,                          # 
        max_block_ms = 3000,                       # Max block time if buffer is full
        max_request_size = 256*1024,               # The maximum size of a request. This is also effectively a cap on the maximum record size. Note that the server has its own cap on record size which may be different from this
        max_in_flight_requests_per_connection = 1, # Requests are pipelined to kafka brokers up to this number of maximum requests per broker connection
        security_protocol = "PLAINTEXT"
    )

    for msg in consumer:
        # -------------
        # Analyze the batch
        # -------------
        processing_time_start = time.perf_counter()
        N_SAMPLES_PER_SCAN = 2048 # Number of complex samples in each scan
        BYTES_PER_SCAN = 8 * N_SAMPLES_PER_SCAN
    
        batch_size = len(msg.value)
        assert batch_size % BYTES_PER_SCAN == 0
        n_scans = batch_size // BYTES_PER_SCAN
    
        # Split raw bytes into I and Q float32 arrays
        i_data = np.frombuffer(msg.value[:batch_size//2], dtype="<f4") # < = little-endian, f4 = float32
        q_data = np.frombuffer(msg.value[batch_size//2:], dtype="<f4")
        complex_signal = np.empty(i_data.shape, dtype=np.complex64) # this way avoids intermediate allocation
        complex_signal.real = i_data
        complex_signal.imag = q_data
    
        # Shape the signal into a matrix to perform vectorized FFT row-wise
        matrix = complex_signal.reshape((n_scans, N_SAMPLES_PER_SCAN))
        # Compute frequency spectrum
        spectra = np.fft.fft(matrix, axis=1)
        power_spectra = np.abs(spectra) ** 2
    
        # Compute mean and second moment of power across scans
        power_means = np.mean(power_spectra, axis=0)
        power_M2s = np.sum((power_spectra - power_means) ** 2, axis=0)
        processing_time = time.perf_counter() - processing_time_start

        # -------------
        # Publish result
        # -------------
        result = {
            "worker_id": worker_id,
            "results": {
                "n_scans": n_scans,
                "power_means": power_means.tolist(),
                "power_M2s": power_M2s.tolist(),
                "producer_ts": msg.timestamp/1000,
                "processing_time" : processing_time,
            } 
        }
        producer.send(output_topic, value=result)
    
    

# Start workers

In [47]:
def report_status(future):
    if future.status == "error":
        print(f"Task {future.key} failed:")
        print(future.exception())
    else:
        print(f"Task {future.key} finished with status {future.status}")

futures = []
for (i, worker_ip) in enumerate(WORKER_IPS):
    future = dask_client.submit(
        worker_function,
        # Parameters to constructor
        worker_id = i + 1,
        bootstrap_servers = BOOTSTRAP_SERVER,
        input_topic = TOPIC_STREAM,
        output_topic = TOPIC_RESULTS,

        # Dask parameters
        key = f"worker{i+1}", # Identifier for the task
        workers = worker_ip,  # Restrict execution to the specific worker
        resources = {},       # no resources (eg GPUs) required
    )
    future.add_done_callback(report_status)
    futures.append(future)

# Close resources

In [48]:
dask_client.close()
dask_cluster.close()

Task worker1 finished with status cancelled
Task worker2 finished with status cancelled
Task worker3 finished with status cancelled
Task worker4 finished with status cancelled
